# Physiology on the spoken-task subset


## 1. Imports and configuration

In [1]:
# Standard library
import gc
import warnings
from collections import OrderedDict
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# --- Reproducibility / protocol constants (matched to physiological_model_tuning.ipynb) ---
RANDOM_STATE = 123
OUTER_SPLITS = 5
INNER_SPLITS = 3
THRESHOLDS = np.arange(0.10, 0.90, 0.02)

# --- Paths (matched to AudioPhysioFusion.ipynb) ---
GT_PATH = "../labels_v2.csv"
PHYSIO_FEATURES_PATH = "physiological_features_50s0overlap_normalized.csv"
AUDIO_PATH = "../audio/audio_features_5s_0overlap_original.csv"

RESULTS_DIR = Path("./results/spoken_subset")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helper functions


In [2]:
def safe_stratified_group_cv(y, groups, max_splits=5):
    y = pd.Series(y).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)

    n_splits = min(
        max_splits,
        int(y.value_counts().min()),
        int(groups.nunique()),
    )

    if n_splits < 2:
        raise ValueError(
            f"Not enough class examples/groups for grouped CV. "
            f"class_counts={y.value_counts().to_dict()}, "
            f"n_groups={groups.nunique()}"
        )

    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


def tune_threshold_cv(y_true, probs, thresholds=THRESHOLDS):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    probs = np.asarray(probs)

    best_thresh = 0.50
    best_bacc = -np.inf

    for thresh in thresholds:
        preds = (probs >= thresh).astype(int)
        bacc = balanced_accuracy_score(y_true, preds)
        if bacc >= best_bacc:
            best_bacc = bacc
            best_thresh = float(thresh)

    return best_thresh


def compute_physio_metrics(y_true, probs, threshold):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    probs = np.asarray(probs)
    preds = (probs >= threshold).astype(int)

    row = {
        "accuracy": accuracy_score(y_true, preds),
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds, zero_division=0),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall_no_stress": recall_score(y_true, preds, pos_label=0, zero_division=0),
        "recall_stress": recall_score(y_true, preds, pos_label=1, zero_division=0),
    }

    row["roc_auc"] = roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan
    return row


def summarize_physio_cv(fold_df, group_cols):
    metric_cols = [
        "accuracy",
        "balanced_accuracy",
        "f1",
        "precision",
        "recall_no_stress",
        "recall_stress",
        "roc_auc",
    ]

    rows = []
    for keys, group in fold_df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for metric in metric_cols:
            row[f"{metric}_mean"] = group[metric].mean()
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        row["n_folds"] = group["fold"].nunique()
        row["n_tasks_mean"] = group["n_test_tasks"].mean()
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(["balanced_accuracy_mean", "f1_mean", "accuracy_mean"], ascending=False)
        .reset_index(drop=True)
    )


In [3]:
def make_physio_model_grids(y_train):
    """Same nested grids as in physiological_model_tuning.ipynb."""
    y_train = pd.Series(y_train).astype(int)
    n_neg = int(y_train.eq(0).sum())
    n_pos = int(y_train.eq(1).sum())
    scale_pos_weight = n_neg / max(n_pos, 1)

    model_grids = OrderedDict()

    model_grids["logreg"] = (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE,
            )),
        ]),
        {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    )

    model_grids["random_forest"] = (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1,
            )),
        ]),
        {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [None, 4, 8],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", 0.5],
        },
    )

    model_grids["balanced_rf"] = (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", BalancedRandomForestClassifier(
                sampling_strategy="auto", replacement=True,
                random_state=RANDOM_STATE, n_jobs=1,
            )),
        ]),
        {
            "clf__n_estimators": [200, 500],
            "clf__max_depth": [None, 4, 8],
            "clf__min_samples_leaf": [1, 2, 4],
            "clf__max_features": ["sqrt", 0.5],
        },
    )

    model_grids["xgboost"] = (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                objective="binary:logistic", eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE, n_jobs=1,
            )),
        ]),
        {
            "clf__n_estimators": [100, 300],
            "clf__max_depth": [2, 3, 4],
            "clf__learning_rate": [0.03, 0.10],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.8, 1.0],
        },
    )

    model_grids["mlp_probe"] = (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                max_iter=1000, early_stopping=True, random_state=RANDOM_STATE,
            )),
        ]),
        {
            "clf__hidden_layer_sizes": [(16,), (32,), (32, 8)],
            "clf__activation": ["relu", "tanh"],
            "clf__alpha": [0.001, 0.01],
            "clf__learning_rate_init": [0.001, 0.005],
        },
    )

    return model_grids


def fit_tuned_model_with_oof_probs(base_model, param_grid, X_train, y_train, groups_train):
    inner_cv = safe_stratified_group_cv(y_train, groups_train, max_splits=INNER_SPLITS)

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )
    grid.fit(X_train, y_train, groups=groups_train)
    best_model = grid.best_estimator_

    train_oof_probs = cross_val_predict(
        clone(best_model),
        X_train, y_train,
        groups=groups_train,
        cv=inner_cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    best_model.fit(X_train, y_train)
    return best_model, train_oof_probs, grid.best_params_, grid.best_score_


In [4]:
def prepare_physio_final_df(df_optimo, final_features):
    df = df_optimo[~df_optimo["activity"].isin(["Baseline", "Relax"])].copy()
    df["target"] = df["target"].astype(int)
    df["subject"] = df["subject"].astype(str)
    df["activity"] = df["activity"].astype(str)
    if "subject/task" not in df.columns:
        df["subject/task"] = df["subject"] + "_" + df["activity"]
    keep = ["subject", "activity", "subject/task", "window_id", "target"] + list(final_features)
    df = df[keep].copy()
    df[final_features] = df[final_features].apply(pd.to_numeric, errors="coerce")
    return df


def evaluate_physio_task_feature_cv(df_optimo, final_features,
                                    outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS):
    df_base = prepare_physio_final_df(df_optimo, final_features)

    agg_dict = {col: ["mean", "std", "max"] for col in final_features}
    agg_dict["target"] = "first"
    agg_dict["subject"] = "first"
    task_df = df_base.groupby("subject/task").agg(agg_dict).reset_index()

    new_cols = []
    for col in task_df.columns.values:
        if col[0] in ["subject/task", "target", "subject"]:
            new_cols.append(col[0])
        else:
            new_cols.append(f"{col[0]}_{col[1]}")
    task_df.columns = new_cols
    task_df = task_df.fillna(0)

    feature_cols = [c for c in task_df.columns if c not in ["subject/task", "target", "subject"]]
    X = task_df[feature_cols]
    y = task_df["target"].astype(int)
    groups = task_df["subject"].astype(str)

    outer_cv = safe_stratified_group_cv(y, groups, max_splits=outer_splits)

    fold_rows = []
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):
        train_subjects = set(groups.iloc[train_idx])
        test_subjects = set(groups.iloc[test_idx])
        assert train_subjects.isdisjoint(test_subjects), "Subject leakage detected."

        X_train = X.iloc[train_idx].reset_index(drop=True)
        X_test = X.iloc[test_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)
        y_test = y.iloc[test_idx].reset_index(drop=True)
        groups_train = groups.iloc[train_idx].reset_index(drop=True)

        for model_name, (base_model, param_grid) in make_physio_model_grids(y_train).items():
            best_model, train_oof_probs, best_params, best_inner_cv_score = (
                fit_tuned_model_with_oof_probs(
                    base_model=base_model,
                    param_grid=param_grid,
                    X_train=X_train,
                    y_train=y_train,
                    groups_train=groups_train,
                )
            )
            threshold = tune_threshold_cv(y_train, train_oof_probs)
            test_probs = best_model.predict_proba(X_test)[:, 1]

            fold_rows.append({
                "modality": "physio",
                "evaluation_stage": "task_feature_aggregation",
                "model": model_name,
                "aggregation": "precomputed_task_features",
                "fold": fold,
                "threshold": threshold,
                "best_inner_cv_score": best_inner_cv_score,
                "best_params": str(best_params),
                "n_train_tasks": len(train_idx),
                "n_test_tasks": len(test_idx),
                "n_train_subjects": len(train_subjects),
                "n_test_subjects": len(test_subjects),
                **compute_physio_metrics(y_test, test_probs, threshold),
            })

    fold_df = pd.DataFrame(fold_rows)
    summary = summarize_physio_cv(
        fold_df,
        group_cols=["modality", "evaluation_stage", "model", "aggregation"],
    )
    return summary, fold_df


def evaluate_physio_window_to_task_cv(df_optimo, final_features,
                                      outer_splits=OUTER_SPLITS, inner_splits=INNER_SPLITS):
    df = prepare_physio_final_df(df_optimo, final_features)

    task_units = (
        df[["subject", "subject/task", "target"]].drop_duplicates().reset_index(drop=True)
    )

    outer_cv = safe_stratified_group_cv(
        task_units["target"], task_units["subject"], max_splits=outer_splits,
    )

    aggregation_strategies = OrderedDict({
        "mean": "mean",
        "median": "median",
        "max": "max",
        "p75": lambda x: np.percentile(x, 75),
    })

    fold_rows = []
    for fold, (task_train_idx, task_test_idx) in enumerate(
        outer_cv.split(task_units, task_units["target"], task_units["subject"]),
        start=1,
    ):
        train_subjects = set(task_units.loc[task_train_idx, "subject"].astype(str))
        test_subjects = set(task_units.loc[task_test_idx, "subject"].astype(str))
        assert train_subjects.isdisjoint(test_subjects), "Subject leakage detected."

        train_window_df = (
            df[df["subject"].isin(train_subjects)]
            .dropna(subset=list(final_features) + ["target", "subject", "subject/task"])
            .reset_index(drop=True)
        )
        test_window_df = (
            df[df["subject"].isin(test_subjects)]
            .dropna(subset=list(final_features) + ["target", "subject", "subject/task"])
            .reset_index(drop=True)
        )

        X_train = train_window_df[final_features].to_numpy()
        y_train = train_window_df["target"].astype(int).to_numpy()
        groups_train = train_window_df["subject"].astype(str).to_numpy()

        X_test = test_window_df[final_features].to_numpy()
        y_test = test_window_df["target"].astype(int).to_numpy()

        for model_name, (base_model, param_grid) in make_physio_model_grids(y_train).items():
            best_model, train_oof_probs, best_params, best_inner_cv_score = (
                fit_tuned_model_with_oof_probs(
                    base_model=base_model,
                    param_grid=param_grid,
                    X_train=X_train,
                    y_train=y_train,
                    groups_train=groups_train,
                )
            )
            test_probs = best_model.predict_proba(X_test)[:, 1]

            train_base = pd.DataFrame({
                "subject/task": train_window_df["subject/task"].to_numpy(),
                "true_label": y_train,
                "pred_prob": train_oof_probs,
            })
            test_base = pd.DataFrame({
                "subject/task": test_window_df["subject/task"].to_numpy(),
                "true_label": y_test,
                "pred_prob": test_probs,
            })

            for aggregation, agg_func in aggregation_strategies.items():
                train_task = (
                    train_base.groupby("subject/task")
                    .agg(true_label=("true_label", "first"), agg_prob=("pred_prob", agg_func))
                    .reset_index()
                )
                test_task = (
                    test_base.groupby("subject/task")
                    .agg(true_label=("true_label", "first"), agg_prob=("pred_prob", agg_func))
                    .reset_index()
                )
                threshold = tune_threshold_cv(train_task["true_label"], train_task["agg_prob"])

                fold_rows.append({
                    "modality": "physio",
                    "evaluation_stage": "window_to_task",
                    "model": model_name,
                    "aggregation": aggregation,
                    "fold": fold,
                    "threshold": threshold,
                    "best_inner_cv_score": best_inner_cv_score,
                    "best_params": str(best_params),
                    "n_train_tasks": train_window_df["subject/task"].nunique(),
                    "n_test_tasks": test_window_df["subject/task"].nunique(),
                    "n_train_subjects": len(train_subjects),
                    "n_test_subjects": len(test_subjects),
                    **compute_physio_metrics(
                        test_task["true_label"], test_task["agg_prob"], threshold
                    ),
                })

    fold_df = pd.DataFrame(fold_rows)
    summary = summarize_physio_cv(
        fold_df,
        group_cols=["modality", "evaluation_stage", "model", "aggregation"],
    )
    return summary, fold_df


## 3. Load physiology features and merge with the ground truth

In [5]:
features_df = pd.read_csv(PHYSIO_FEATURES_PATH)
print(f"Loaded physiology features: shape = {features_df.shape}")

gt = pd.read_csv(GT_PATH, sep=";")

features_df["subject/task"] = (
    features_df["subject"].astype(str) + "_" + features_df["activity"].astype(str)
)
df_full = features_df.merge(
    gt[["subject/task", "binary-stress"]], on="subject/task", how="inner",
).rename(columns={"binary-stress": "target"})

# Define the feature columns: everything that is not metadata or label.
META_COLS = {"subject", "activity", "window_id", "subject/task", "target"}
final_features = [c for c in df_full.columns if c not in META_COLS]
print(f"Number of physiology features used: {len(final_features)}")
print(final_features)


Loaded physiology features: shape = (1107, 18)
Number of physiology features used: 15
['ECG_Rate_Mean', 'HRV_MedianNN', 'HRV_Prc20NN', 'HRV_Prc80NN', 'HRV_MaxNN', 'HRV_PSS', 'HRV_C1d', 'HRV_SampEn', 'HRV_MSEn', 'SCR_Peaks_Amplitude_Mean', 'RSP_Rate_Mean', 'RAV_SD', 'RAV_RMSSD', 'RSP_RVT', 'RSP_Symmetry_RiseDecay']


## 4. Define the spoken subset


In [6]:
audio_df = pd.read_csv(AUDIO_PATH, usecols=["subject_activity"])
spoken_keys = set(audio_df["subject_activity"].astype(str).unique())
print(f"Number of spoken (audio-available) task keys: {len(spoken_keys)}")

# Intersect with the physiology task keys we actually have.
physio_keys = set(df_full["subject/task"].astype(str).unique())
matched_keys = spoken_keys & physio_keys
print(f"Number of spoken tasks also present in physiology: {len(matched_keys)}")

df_spoken = df_full[df_full["subject/task"].isin(matched_keys)].copy()


Number of spoken (audio-available) task keys: 378
Number of spoken tasks also present in physiology: 368


### Sanity check: subset composition

Compare the two evaluation sets in terms of unique tasks (after removing the
`Baseline` / `Relax` normalisation activities used inside the evaluation
helpers), number of subjects, and class balance. This lets us interpret any
performance gap: a shift driven by class balance is very different from one
driven by the harder task mix.


In [7]:
def describe_subset(df, name):
    df_eval = df[~df["activity"].isin(["Baseline", "Relax"])].copy()
    task_level = (
        df_eval[["subject/task", "subject", "activity", "target"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    rows = {
        "subset": name,
        "n_windows": len(df_eval),
        "n_tasks": task_level["subject/task"].nunique(),
        "n_subjects": task_level["subject"].nunique(),
        "n_activities": task_level["activity"].nunique(),
        "frac_stress_tasks": task_level["target"].mean(),
    }
    return rows, task_level


full_row, full_tasks = describe_subset(df_full, "full")
spoken_row, spoken_tasks = describe_subset(df_spoken, "spoken")

composition_df = pd.DataFrame([full_row, spoken_row])
display(composition_df)

print("\nActivities present in spoken subset:")
display(
    spoken_tasks.groupby("activity")
    .agg(n_tasks=("subject/task", "nunique"),
         n_subjects=("subject", "nunique"),
         frac_stress=("target", "mean"))
    .sort_values("n_tasks", ascending=False)
)


,subset,n_windows,n_tasks,n_subjects,n_activities,frac_stress_tasks
0,full,867,606,62,10,0.580858
1,spoken,368,368,53,7,0.725543



Activities present in spoken subset:


,n_tasks,n_subjects,frac_stress
activity,,,
Counting1,53,53,0.716981
Counting2,53,53,0.792453
Math,53,53,0.698113
Speaking,53,53,0.735849
Reading,53,53,0.566038
Stroop,52,52,0.807692
Counting3,51,51,0.764706


## 5.CV evaluation

Each call below runs:

- `evaluate_physio_task_feature_cv` -> aggregate 50 s features into one
  task-level vector (mean / std / max per feature) and classify directly at
  the task level.
- `evaluate_physio_window_to_task_cv` -> classify 50 s windows, then aggregate
  the per-window probabilities (mean / median / max / p75) into a task-level
  decision.

For each outer fold we run an inner grid search (logreg, random forest,
balanced RF, XGBoost, MLP probe), pick the model+hyperparameters that
maximise inner balanced accuracy, tune the decision threshold on outer-train
OOF probabilities, and finally evaluate on the held-out outer test fold.


In [8]:
print("=" * 70)
print(" FULL dataset: task-feature aggregation")
print("=" * 70)
full_task_summary, full_task_folds = evaluate_physio_task_feature_cv(
    df_full, final_features,
)

print("\n" + "=" * 70)
print(" FULL dataset: window-to-task aggregation")
print("=" * 70)
full_window_summary, full_window_folds = evaluate_physio_window_to_task_cv(
    df_full, final_features,
)

full_summary = pd.concat([full_task_summary, full_window_summary], ignore_index=True)
full_summary["subset"] = "full"

full_folds = pd.concat([full_task_folds, full_window_folds], ignore_index=True)
full_folds["subset"] = "full"

display(
    full_summary.sort_values(
        ["balanced_accuracy_mean", "f1_mean"], ascending=False,
    ).head(15)
)


 FULL dataset: task-feature aggregation

 FULL dataset: window-to-task aggregation


,modality,evaluation_stage,model,aggregation,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std,...,precision_std,recall_no_stress_mean,recall_no_stress_std,recall_stress_mean,recall_stress_std,roc_auc_mean,roc_auc_std,n_folds,n_tasks_mean,subset
0,physio,task_feature_aggregation,mlp_probe,precomputed_task_features,0.671555,0.050787,0.645665,0.053188,0.740141,0.041936,...,0.043842,0.484094,0.091665,0.807236,0.062816,0.691432,0.089579,5,121.2,full
1,physio,task_feature_aggregation,balanced_rf,precomputed_task_features,0.670852,0.052623,0.640954,0.045651,0.741397,0.054111,...,0.029750,0.460778,0.059429,0.821130,0.092392,0.676238,0.041484,5,121.2,full
5,physio,window_to_task,xgboost,median,0.646989,0.059803,0.640119,0.061137,0.692293,0.053957,...,0.057975,0.594383,0.091589,0.685855,0.068123,0.689290,0.046338,5,121.2,full
6,physio,window_to_task,xgboost,mean,0.645258,0.068098,0.635578,0.069797,0.695465,0.060918,...,0.066066,0.570465,0.109671,0.700692,0.082635,0.686128,0.049788,5,121.2,full
2,physio,task_feature_aggregation,random_forest,precomputed_task_features,0.651691,0.033898,0.630379,0.031858,0.715774,0.041692,...,0.028051,0.499620,0.079293,0.761137,0.080089,0.679476,0.039718,5,121.2,full
3,physio,task_feature_aggregation,xgboost,precomputed_task_features,0.644864,0.036415,0.628850,0.035362,0.703851,0.039212,...,0.038830,0.526845,0.079811,0.730855,0.072866,0.690476,0.044669,5,121.2,full
7,physio,window_to_task,random_forest,median,0.636621,0.029823,0.625656,0.026842,0.687255,0.037098,...,0.028576,0.559608,0.065795,0.691705,0.067144,0.681583,0.049404,5,121.2,full
8,physio,window_to_task,random_forest,mean,0.636686,0.030791,0.624335,0.034008,0.688946,0.033736,...,0.037241,0.552311,0.100482,0.696360,0.066405,0.679616,0.052387,5,121.2,full
9,physio,window_to_task,balanced_rf,mean,0.628516,0.046987,0.623365,0.043339,0.668841,0.059832,...,0.048788,0.590386,0.110070,0.656344,0.112055,0.676414,0.056297,5,121.2,full
10,physio,window_to_task,xgboost,p75,0.637552,0.058057,0.623071,0.061213,0.695277,0.050345,...,0.055517,0.531858,0.112546,0.714283,0.074581,0.675481,0.052235,5,121.2,full


In [9]:
print("=" * 70)
print(" SPOKEN subset: task-feature aggregation")
print("=" * 70)
spoken_task_summary, spoken_task_folds = evaluate_physio_task_feature_cv(
    df_spoken, final_features,
)

print("\n" + "=" * 70)
print(" SPOKEN subset: window-to-task aggregation")
print("=" * 70)
spoken_window_summary, spoken_window_folds = evaluate_physio_window_to_task_cv(
    df_spoken, final_features,
)

spoken_summary = pd.concat(
    [spoken_task_summary, spoken_window_summary], ignore_index=True,
)
spoken_summary["subset"] = "spoken"

spoken_folds = pd.concat([spoken_task_folds, spoken_window_folds], ignore_index=True)
spoken_folds["subset"] = "spoken"

display(
    spoken_summary.sort_values(
        ["balanced_accuracy_mean", "f1_mean"], ascending=False,
    ).head(15)
)


 SPOKEN subset: task-feature aggregation

 SPOKEN subset: window-to-task aggregation


,modality,evaluation_stage,model,aggregation,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std,...,precision_std,recall_no_stress_mean,recall_no_stress_std,recall_stress_mean,recall_stress_std,roc_auc_mean,roc_auc_std,n_folds,n_tasks_mean,subset
0,physio,task_feature_aggregation,logreg,precomputed_task_features,0.567031,0.080184,0.578264,0.085056,0.645679,0.104419,...,0.081473,0.585270,0.284320,0.571258,0.168996,0.605320,0.146948,5,73.6,spoken
5,physio,window_to_task,logreg,max,0.553656,0.087090,0.566775,0.060231,0.629597,0.129127,...,0.081971,0.574844,0.253507,0.558706,0.181156,0.599230,0.143632,5,73.6,spoken
6,physio,window_to_task,logreg,mean,0.553656,0.087090,0.566775,0.060231,0.629597,0.129127,...,0.081971,0.574844,0.253507,0.558706,0.181156,0.599230,0.143632,5,73.6,spoken
7,physio,window_to_task,logreg,median,0.553656,0.087090,0.566775,0.060231,0.629597,0.129127,...,0.081971,0.574844,0.253507,0.558706,0.181156,0.599230,0.143632,5,73.6,spoken
8,physio,window_to_task,logreg,p75,0.553656,0.087090,0.566775,0.060231,0.629597,0.129127,...,0.081971,0.574844,0.253507,0.558706,0.181156,0.599230,0.143632,5,73.6,spoken
9,physio,window_to_task,balanced_rf,max,0.469685,0.108582,0.550164,0.029325,0.459557,0.250514,...,0.098833,0.734571,0.207980,0.365758,0.225074,0.548763,0.072667,5,73.6,spoken
10,physio,window_to_task,balanced_rf,mean,0.469685,0.108582,0.550164,0.029325,0.459557,0.250514,...,0.098833,0.734571,0.207980,0.365758,0.225074,0.548763,0.072667,5,73.6,spoken
11,physio,window_to_task,balanced_rf,median,0.469685,0.108582,0.550164,0.029325,0.459557,0.250514,...,0.098833,0.734571,0.207980,0.365758,0.225074,0.548763,0.072667,5,73.6,spoken
12,physio,window_to_task,balanced_rf,p75,0.469685,0.108582,0.550164,0.029325,0.459557,0.250514,...,0.098833,0.734571,0.207980,0.365758,0.225074,0.548763,0.072667,5,73.6,spoken
13,physio,window_to_task,random_forest,max,0.449727,0.087732,0.545336,0.031997,0.428071,0.212058,...,0.097151,0.770754,0.189728,0.319918,0.183890,0.581852,0.078164,5,73.6,spoken


## 6. Side-by-side comparison

The next cell merges the full and spoken summaries on
`(evaluation_stage, model, aggregation)` and reports the delta on the key
metrics. A negative delta means performance drops on the spoken subset; a
positive one means it improves.


In [10]:
METRIC_COLS_REPORT = [
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "f1_mean", "f1_std",
    "accuracy_mean", "accuracy_std",
    "recall_stress_mean", "recall_no_stress_mean",
    "roc_auc_mean",
]
JOIN_COLS = ["modality", "evaluation_stage", "model", "aggregation"]

cmp_df = full_summary.merge(
    spoken_summary,
    on=JOIN_COLS,
    suffixes=("_full", "_spoken"),
    how="inner",
)

# Compute deltas for the main metrics.
delta_metrics = [
    "balanced_accuracy_mean",
    "f1_mean",
    "accuracy_mean",
    "recall_stress_mean",
    "recall_no_stress_mean",
    "roc_auc_mean",
]
for m in delta_metrics:
    cmp_df[f"delta_{m}"] = cmp_df[f"{m}_spoken"] - cmp_df[f"{m}_full"]

display_cols = JOIN_COLS + [
    "balanced_accuracy_mean_full", "balanced_accuracy_std_full",
    "balanced_accuracy_mean_spoken", "balanced_accuracy_std_spoken",
    "delta_balanced_accuracy_mean",
    "f1_mean_full", "f1_mean_spoken", "delta_f1_mean",
    "n_tasks_mean_full", "n_tasks_mean_spoken",
]

cmp_df_display = cmp_df[display_cols].sort_values(
    "balanced_accuracy_mean_spoken", ascending=False,
)
display(cmp_df_display.head(20))


,modality,evaluation_stage,model,aggregation,balanced_accuracy_mean_full,balanced_accuracy_std_full,balanced_accuracy_mean_spoken,balanced_accuracy_std_spoken,delta_balanced_accuracy_mean,f1_mean_full,f1_mean_spoken,delta_f1_mean,n_tasks_mean_full,n_tasks_mean_spoken
4,physio,task_feature_aggregation,logreg,precomputed_task_features,0.621418,0.057927,0.578264,0.085056,-0.043154,0.684388,0.645679,-0.038709,121.2,73.6
19,physio,window_to_task,logreg,mean,0.603717,0.072492,0.566775,0.060231,-0.036942,0.651216,0.629597,-0.021619,121.2,73.6
21,physio,window_to_task,logreg,p75,0.599784,0.061739,0.566775,0.060231,-0.033009,0.637643,0.629597,-0.008046,121.2,73.6
24,physio,window_to_task,logreg,max,0.581908,0.052810,0.566775,0.060231,-0.015133,0.535702,0.629597,0.093895,121.2,73.6
15,physio,window_to_task,logreg,median,0.607367,0.066547,0.566775,0.060231,-0.040593,0.637846,0.629597,-0.008249,121.2,73.6
11,physio,window_to_task,balanced_rf,median,0.622913,0.038344,0.550164,0.029325,-0.072749,0.670119,0.459557,-0.210562,121.2,73.6
13,physio,window_to_task,balanced_rf,p75,0.615140,0.043387,0.550164,0.029325,-0.064976,0.667851,0.459557,-0.208294,121.2,73.6
9,physio,window_to_task,balanced_rf,mean,0.623365,0.043339,0.550164,0.029325,-0.073201,0.668841,0.459557,-0.209284,121.2,73.6
16,physio,window_to_task,balanced_rf,max,0.606471,0.053965,0.550164,0.029325,-0.056307,0.654023,0.459557,-0.194467,121.2,73.6
12,physio,window_to_task,random_forest,p75,0.615228,0.030678,0.545336,0.031997,-0.069892,0.679802,0.428071,-0.251731,121.2,73.6


### Top configurations


In [11]:
def headline_per_stage(summary, subset_name):
    rows = []
    for stage, group in summary.groupby("evaluation_stage"):
        top = group.sort_values("balanced_accuracy_mean", ascending=False).iloc[0]
        rows.append({
            "subset": subset_name,
            "evaluation_stage": stage,
            "model": top["model"],
            "aggregation": top["aggregation"],
            "balanced_accuracy_mean": top["balanced_accuracy_mean"],
            "balanced_accuracy_std": top["balanced_accuracy_std"],
            "f1_mean": top["f1_mean"],
            "f1_std": top["f1_std"],
            "accuracy_mean": top["accuracy_mean"],
            "recall_stress_mean": top["recall_stress_mean"],
            "recall_no_stress_mean": top["recall_no_stress_mean"],
            "roc_auc_mean": top["roc_auc_mean"],
            "n_tasks_mean": top["n_tasks_mean"],
        })
    return pd.DataFrame(rows)


headline = pd.concat(
    [headline_per_stage(full_summary, "full"),
     headline_per_stage(spoken_summary, "spoken")],
    ignore_index=True,
).sort_values(["evaluation_stage", "subset"]).reset_index(drop=True)

display(headline)


,subset,evaluation_stage,model,aggregation,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std,accuracy_mean,recall_stress_mean,recall_no_stress_mean,roc_auc_mean,n_tasks_mean
0,full,task_feature_aggregation,mlp_probe,precomputed_task_features,0.645665,0.053188,0.740141,0.041936,0.671555,0.807236,0.484094,0.691432,121.2
1,spoken,task_feature_aggregation,logreg,precomputed_task_features,0.578264,0.085056,0.645679,0.104419,0.567031,0.571258,0.585270,0.605320,73.6
2,full,window_to_task,xgboost,median,0.640119,0.061137,0.692293,0.053957,0.646989,0.685855,0.594383,0.689290,121.2
3,spoken,window_to_task,logreg,max,0.566775,0.060231,0.629597,0.129127,0.553656,0.558706,0.574844,0.599230,73.6


In [ ]:
full_summary.to_csv(RESULTS_DIR / "physio_full_subset_cv_summary.csv", index=False)
spoken_summary.to_csv(RESULTS_DIR / "physio_spoken_subset_cv_summary.csv", index=False)

full_folds.to_csv(RESULTS_DIR / "physio_full_subset_cv_folds.csv", index=False)
spoken_folds.to_csv(RESULTS_DIR / "physio_spoken_subset_cv_folds.csv", index=False)

cmp_df.to_csv(RESULTS_DIR / "physio_full_vs_spoken_comparison.csv", index=False)
headline.to_csv(RESULTS_DIR / "physio_full_vs_spoken_headline.csv", index=False)
composition_df.to_csv(RESULTS_DIR / "physio_subset_composition.csv", index=False)

All results saved under: /home/ethe/Documents/TFG/multimodal_stress_detection/physiological/results/spoken_subset
